# Training Neural Networks to Infer the Peclet Number

This notebook trains convolutional neural networks to infer various
parameters of 2D streampower-diffusion landscape evolution models from
the output topography or topographic derivatives. Of particular note are
the following parameters:

`DATA_TYPES`  
The various data types (elevation, slope, etc) to use as an input

`LABELS`  
The various targets ($D/K$, $D/K$, $\log_{10}(D/K)$, etc) for the neural
network

In [ ]:
import torch
from neural_spd.train_peclet_model import PecletModelTrainer
from neural_spd.ThreeLayerCNNRegressor import ThreeLayerCNNRegressor, JumboThreeLayerCNNRegressor
import json
import os
import numpy as np

from pathlib import Path
from neural_spd.config import MODEL_STATS_PATH, DB_PATH, MODEL_DEM_PATH, MODEL_SLOPE_PATH, MODEL_ACC_PATH, MODEL_CURV_PATH, WEIGHTS_PATH, NN_SEEDS, NUM_EPOCHS, LEARNING_RATE, RETRAIN_MODELS, NOISE_LEVELS, DATA_TYPES, LABELS, DATA_PATH, LOG_PATH, IS_HEADLESS, CHECKPOINT_PATH, BATCH_SIZE
import os

# Use environment variables if set (for HPC scratch filesystem)
DATA_PATH = Path(os.getenv('DATA_PATH', DATA_PATH))
WEIGHTS_PATH = Path(os.getenv('WEIGHTS_PATH', WEIGHTS_PATH))
MODEL_STATS_PATH = DATA_PATH / "model_stats.json"
MODEL_STATS_PATH = Path(os.getenv('MODEL_STATS_PATH', MODEL_STATS_PATH))
DB_PATH = DATA_PATH / "model_runs.db"
DB_PATH = Path(os.getenv('DB_PATH', DB_PATH))
from itertools import product


If this is being run in Colab, we want to copy over the data to the
local Colab instance. If you are using the big tangled colab script,
this has probably already been done. Maybe this should be removed.

In [ ]:
from neural_spd.config import IN_COLAB
if IN_COLAB:
    !cp -r {DATA_PATH} /content/data
    DATA_PATH = Path("/content/data")


In [ ]:
with open(MODEL_STATS_PATH, 'r') as f:
    statistics = json.load(f)


This function trains a neural network for a given parameter combination.
The `PecletModelTrainer` object saves checkpoints, and this function
will reload from that checkpoint (if existing). This can be helpful,
since the training 80+ neural networks can sometimes get interrupted.

In [ ]:
def train_neural_net(seed, noise, data_type, label, reload_from_checkpoint=True):
    label_key, label_query = label
    torch.manual_seed(seed)
    weights_path = WEIGHTS_PATH / f"n{str(noise).replace('.', '-')}_{data_type}_{seed}_{label_key}_weights.pt"
    log_path = LOG_PATH / f"n{str(noise).replace('.', '-')}_{data_type}_{seed}_{label_key}_training_log.json"
    dataset_path = DATA_PATH / str(noise).replace('.', '-') / data_type
    checkpoint_path = CHECKPOINT_PATH / f"n{str(noise).replace('.', '-')}_{data_type}_{seed}_{label_key}_checkpoint.pt"
    if not weights_path.exists() or RETRAIN_MODELS:
        print(f"Training {weights_path}")
        label_stats = statistics[label_key]
        data_stats = statistics[str(noise).replace('.', '-')][data_type]
        trainer = PecletModelTrainer(DB_PATH,
                                    dataset_path,
                                    ThreeLayerCNNRegressor(),
                                    label_query,
                                    epochs=NUM_EPOCHS,
                                    learning_rate=LEARNING_RATE,
                                    batch_size=BATCH_SIZE,
                                    **data_stats,
                                    **label_stats)
        trainer.train(checkpoint_path=checkpoint_path, reload_from_checkpoint=reload_from_checkpoint)
        trainer.save_weights(weights_path)
        trainer.save_training_history(log_path)
    else:
        print(f"{weights_path} exists, skipping")



This creates all possible combinations and runs this. This is really
only useful when being run as from the SLURM script, which allows it to
run an arbitrary parameter combination that corresponds to a specific
SLURM job. This could be adapated to use Python mulitprocessing for
running locally, but it hasn't.

In [ ]:
runs = list(product(NN_SEEDS, NOISE_LEVELS, DATA_TYPES, LABELS.items()))
if IS_HEADLESS:
    task_id = int(os.environ.get("SLURM_ARRAY_TASK_ID", 0))
    train_neural_net(*runs[task_id])
else:
    for run in runs:
        train_neural_net(*run)
